In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('../data/master/eda_output.csv')
print(df.shape)
df[['income_group', 'region', 'new_business_density', 'log_new_business_density', 'governance_index']].head()

C:\Users\AJAY\AppData\Roaming\Python\Python310\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


(3477, 51)


,income_group,region,new_business_density,log_new_business_density,governance_index
0,Low income,"Middle East, North Africa, Afghanistan & Pakistan",NaN,NaN,-3.869491
1,Low income,"Middle East, North Africa, Afghanistan & Pakistan",NaN,NaN,-3.941579
2,Low income,"Middle East, North Africa, Afghanistan & Pakistan",NaN,NaN,-4.175190
3,Low income,"Middle East, North Africa, Afghanistan & Pakistan",0.272384,0.240892,-4.100728
4,Low income,"Middle East, North Africa, Afghanistan & Pakistan",0.341805,0.294016,-4.170993


In [2]:
print(f"Total rows: {len(df)}")
print(f"Rows with non-null target: {df['new_business_density'].notnull().sum()}")
print(f"Rows with null target: {df['new_business_density'].isnull().sum()}")

Total rows: 3477
Rows with non-null target: 2911
Rows with null target: 566


In [3]:
income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
df_target = df.dropna(subset=['log_new_business_density', 'income_group'])

# Normality check per group (Shapiro-Wilk) -- sample if group is large, since Shapiro is sensitive to big n
print("Shapiro-Wilk normality test (log_new_business_density) per income group:")
for grp in income_order:
    vals = df_target[df_target['income_group'] == grp]['log_new_business_density']
    sample = vals.sample(min(len(vals), 500), random_state=42)  # cap at 500 for test stability
    stat, p = stats.shapiro(sample)
    print(f"{grp}: n={len(vals)}, W={stat:.4f}, p={p:.4g}")

Shapiro-Wilk normality test (log_new_business_density) per income group:
Low income: n=271, W=0.8184, p=4.093e-17
Lower middle income: n=652, W=0.8704, p=5.677e-20
Upper middle income: n=918, W=0.9408, p=3.257e-13
High income: n=1070, W=0.9303, p=1.686e-14


In [4]:
# Variance homogeneity across groups (Levene's test)
groups = [df_target[df_target['income_group'] == g]['log_new_business_density'] for g in income_order]
stat, p = stats.levene(*groups)
print(f"\nLevene's test: statistic={stat:.4f}, p={p:.4g}")


Levene's test: statistic=87.8628, p=2.022e-54


In [5]:
# Kruskal-Wallis: non-parametric alternative to ANOVA
groups = [df_target[df_target['income_group'] == g]['log_new_business_density'] for g in income_order]
stat, p = stats.kruskal(*groups)
print(f"Kruskal-Wallis: H-statistic={stat:.4f}, p={p:.4g}")

Kruskal-Wallis: H-statistic=1295.4331, p=1.441e-280


In [6]:
# Effect size for Kruskal-Wallis: epsilon-squared
n = len(df_target)
k = len(income_order)
epsilon_sq = (stat - k + 1) / (n - k)
print(f"Epsilon-squared (effect size): {epsilon_sq:.4f}")

Epsilon-squared (effect size): 0.4446


In [7]:
# Dunn's test requires scikit-posthocs
import scikit_posthocs as sp

dunn_result = sp.posthoc_dunn(df_target, val_col='log_new_business_density', 
                                group_col='income_group', p_adjust='bonferroni')
print(dunn_result.round(4))

                     High income  Low income  Lower middle income  \
High income                  1.0         0.0                  0.0   
Low income                   0.0         1.0                  0.0   
Lower middle income          0.0         0.0                  1.0   
Upper middle income          0.0         0.0                  0.0   

                     Upper middle income  
High income                          0.0  
Low income                           0.0  
Lower middle income                  0.0  
Upper middle income                  1.0  


## Hypothesis Test: Income Group vs. Business Density

**H0**: Business density distribution is the same across all income groups.

**H1**: At least one income group differs.

- Normality (Shapiro-Wilk) and variance homogeneity (Levene's) both failed for all groups → used non-parametric Kruskal-Wallis instead of ANOVA.
- **Kruskal-Wallis**: H=1295.43, p<0.0001 → reject H0, groups differ significantly.
- **Effect size (ε²=0.4446)**: income group explains ~44% of variance in business density — a large effect, not just statistically significant.
- **Post-hoc (Dunn's test, Bonferroni-corrected)**: all 6 pairwise comparisons significant (p<0.0001) — every income group is distinct from every other.

**Conclusion**: Income group has a strong, robust, monotonic relationship with new business formation, confirmed by both significance and effect size.

In [10]:
df_gov = df.dropna(subset=['governance_index', 'log_new_business_density'])
print(f"n = {len(df_gov)}")

# Spearman is safer than Pearson here since target isn't normally distributed even after log transform
r, p = stats.spearmanr(df_gov['governance_index'], df_gov['log_new_business_density'])
print(f"Spearman correlation: r={r:.4f}, p={p:.4g}")

n = 2793
Spearman correlation: r=0.6801, p=0


In [11]:
# Confidence interval for the correlation via Fisher z-transformation
n = len(df_gov)
z = np.arctanh(r)
se = 1 / np.sqrt(n - 3)
ci_low, ci_high = np.tanh(z - 1.96*se), np.tanh(z + 1.96*se)
print(f"95% CI for correlation: ({ci_low:.4f}, {ci_high:.4f})")

95% CI for correlation: (0.6596, 0.6995)


## Hypothesis Test: Governance Index vs. Business Density

**H0**: No monotonic relationship between governance_index and log_new_business_density.
**H1**: A monotonic relationship exists.

- Used Spearman's rank correlation (robust to non-normality, unlike Pearson).
- **r=0.680, p<0.0001** (n=2793) → reject H0, strong positive relationship confirmed.
- **95% CI: (0.660, 0.700)** — tight interval, relationship is precisely estimated, not fragile.

**Conclusion**: Governance quality has a strong, statistically robust, positive relationship 
with new business formation — the strongest single predictor identified in this analysis.

In [12]:
regions = df['region'].unique().tolist()
df_region = df.dropna(subset=['log_new_business_density', 'region'])

print("Shapiro-Wilk per region (sampled):")
for r in regions:
    vals = df_region[df_region['region'] == r]['log_new_business_density']
    if len(vals) < 3:
        print(f"{r}: n={len(vals)}, skipped (too few observations)")
        continue
    sample = vals.sample(min(len(vals), 500), random_state=42)
    stat, p = stats.shapiro(sample)
    print(f"{r}: n={len(vals)}, W={stat:.4f}, p={p:.4g}")

Shapiro-Wilk per region (sampled):
Middle East, North Africa, Afghanistan & Pakistan: n=303, W=0.9092, p=1.498e-12
Sub-Saharan Africa: n=671, W=0.8543, p=4.141e-21
Europe & Central Asia: n=902, W=0.9753, p=1.838e-07
Latin America & Caribbean: n=493, W=0.7848, p=5.546e-25
East Asia & Pacific: n=438, W=0.8756, p=2.871e-18
South Asia: n=98, W=0.7503, p=1.293e-11
North America: n=6, W=0.6985, p=0.005955


In [13]:
groups = [df_region[df_region['region'] == r]['log_new_business_density'] for r in regions if len(df_region[df_region['region']==r]) >= 3]
stat, p = stats.kruskal(*groups)
print(f"\nKruskal-Wallis: H={stat:.4f}, p={p:.4g}")

n = len(df_region)
k = len(groups)
epsilon_sq = (stat - k + 1) / (n - k)
print(f"Epsilon-squared: {epsilon_sq:.4f}")


Kruskal-Wallis: H=594.6545, p=3.317e-125
Epsilon-squared: 0.2027


In [14]:
offshore_vals = df_region[df_region['is_offshore_center']]['log_new_business_density']
non_offshore_vals = df_region[~df_region['is_offshore_center']]['log_new_business_density']

stat, p = stats.mannwhitneyu(offshore_vals, non_offshore_vals, alternative='two-sided')
print(f"Mann-Whitney U: statistic={stat:.4f}, p={p:.4g}")
print(f"Offshore median: {offshore_vals.median():.4f}, n={len(offshore_vals)}")
print(f"Non-offshore median: {non_offshore_vals.median():.4f}, n={len(non_offshore_vals)}")

# Effect size: rank-biserial correlation
r_effect = 1 - (2*stat) / (len(offshore_vals) * len(non_offshore_vals))
print(f"Rank-biserial effect size: {r_effect:.4f}")

Mann-Whitney U: statistic=109163.0000, p=2.882e-26
Offshore median: 4.7616, n=38
Non-offshore median: 0.9850, n=2873
Rank-biserial effect size: -0.9998


## Hypothesis Tests: Region & Offshore Centers

**Region** (Kruskal-Wallis): H=594.65, p<0.0001, ε²=0.203 — regions differ significantly, 
large effect, but weaker than income group (ε²=0.44), reflecting income diversity within regions.

**Offshore financial centers** (Mann-Whitney U): p<0.0001, rank-biserial r=-0.9998 (near-perfect 
separation). Offshore median density (4.76) is ~5x non-offshore median (0.99). Confirms offshore 
centers are a statistically distinct population, not just high-variance noise — justifies keeping 
them flagged rather than dropped or blended into general modeling.

## Overall Statistical Analysis Conclusion
All three hypothesized relationships (income group, governance, region) are statistically 
significant with meaningful effect sizes, using non-parametric tests throughout since normality 
and variance-homogeneity assumptions failed for all groupings. Governance quality (r=0.68) and 
income group (ε²=0.44) are the two strongest, most robust drivers of new business density 
identified in this analysis. Offshore financial centers are confirmed as a distinct population 
requiring separate treatment in modeling.